"""
Ingestão do relatório de segurança pública -> vetores, com tratamento
DEDICADO para tabelas.

Por que mudar em relação ao PyPDFDirectoryLoader "puro":
- PyPDFDirectoryLoader extrai só o texto corrido da página. Tabelas viram uma
  sequência de números/palavras sem separação de coluna -> o embedding não
  consegue associar "12.345" à coluna "Roubos" e à linha "2023".
- Aqui usamos pdfplumber por página:
    1. Extrai as tabelas separadamente e converte cada uma em Markdown
       (mantém cabeçalho de coluna + linha, vira um Document com
       metadata tipo="tabela").
    2. O texto restante da página (fora das tabelas) segue pro
       RecursiveCharacterTextSplitter normal, igual ao pipeline antigo
       (metadata tipo="texto").
    3. Tabela NUNCA é cortada pelo splitter de tamanho fixo — ou fica
       inteira num chunk, ou (se for muito grande) é dividida repetindo
       o cabeçalho em cada pedaço, pra nunca perder o significado das
       colunas.

Requisito adicional: pip install pdfplumber
"""

## imports

In [ ]:
import os

import re
import time
from pathlib import Path
from dotenv import load_dotenv

import pdfplumber
from tqdm import tqdm
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import warnings
warnings.filterwarnings('ignore')

OUTPUT_DOCUMENTS_DIR = Path("./data/fonte/")
VECTORSTORE_DIR = Path("./data/vectorstore_teste")



ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

print('✔ OUTPUT_DOCUMENTS_DIR:',OUTPUT_DOCUMENTS_DIR)
carrega_variaveis_ambiente()


TITULO_TABELA_REGEX = re.compile(r"(tabela|quadro|gr[aá]fico)\s*\d+", re.IGNORECASE)
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


## Funções

In [3]:
# ==========================================
# 1. EXTRAÇÃO — texto e tabelas separados
# ==========================================
def extrair_titulo_tabela(linhas_antes: list[str]) -> str:
    """Procura, nas últimas linhas antes da tabela, algo como 'Tabela 3 - Ocorrências...'."""
    for linha in reversed(linhas_antes[-5:]):
        if TITULO_TABELA_REGEX.search(linha):
            return linha.strip()
    return ""


def tabela_para_markdown(tabela: list[list]) -> str:
    """Converte uma tabela extraída pelo pdfplumber (lista de linhas) em Markdown."""
    linhas_limpas = [
        [(cel or "").strip().replace("\n", " ") for cel in linha]
        for linha in tabela
        if any(cel for cel in linha)
    ]
    if len(linhas_limpas) < 2:
        return ""

    cabecalho = linhas_limpas[0]
    corpo = linhas_limpas[1:]

    md = "| " + " | ".join(cabecalho) + " |\n"
    md += "|" + "|".join(["---"] * len(cabecalho)) + "|\n"
    for linha in corpo:
        linha_ajustada = (linha + [""] * len(cabecalho))[:len(cabecalho)]
        md += "| " + " | ".join(linha_ajustada) + " |\n"

    return md


def processar_pdf(caminho_pdf: Path) -> tuple[list[Document], list[Document]]:
    """Retorna (documentos_texto, documentos_tabela) de um único PDF."""
    docs_texto, docs_tabela = [], []

    with pdfplumber.open(caminho_pdf) as pdf:
        for num_pagina, pagina in enumerate(pdf.pages):
            texto_completo = pagina.extract_text() or ""
            linhas = texto_completo.split("\n")

            tabelas = pagina.find_tables()
            for idx_tab, tabela in enumerate(tabelas):
                dados = tabela.extract()
                md_tabela = tabela_para_markdown(dados)
                if not md_tabela:
                    continue

                titulo = extrair_titulo_tabela(linhas)

                docs_tabela.append(Document(
                    page_content=md_tabela,
                    metadata={
                        "source": str(caminho_pdf),
                        "page": num_pagina,
                        "tipo": "tabela",
                        "titulo_tabela": titulo or f"Tabela sem título (pág. {num_pagina + 1})",
                        "tabela_idx_na_pagina": idx_tab,
                    }
                ))

            # Nota: o texto da página continua incluindo os números da tabela
            # (o pdfplumber não remove automaticamente). Isso é aceitável:
            # o BM25/vetor pode até achar esse texto, mas a resposta "boa"
            # vem do chunk tipo="tabela", que tem estrutura de coluna.
            if texto_completo.strip():
                docs_texto.append(Document(
                    page_content=texto_completo,
                    metadata={
                        "source": str(caminho_pdf),
                        "page": num_pagina,
                        "tipo": "texto",
                    }
                ))

    return docs_texto, docs_tabela


def carregar_documentos(diretorio: Path) -> tuple[list[Document], list[Document]]:
    todos_texto, todas_tabelas = [], []
    pdfs = sorted(diretorio.glob("*.pdf"))
    print(f"✔ {len(pdfs)} PDF(s) encontrado(s) em {diretorio}")

    for pdf_path in pdfs:
        docs_texto, docs_tabela = processar_pdf(pdf_path)
        todos_texto.extend(docs_texto)
        todas_tabelas.extend(docs_tabela)
        print(f"  - {pdf_path.name}: {len(docs_texto)} páginas de texto | {len(docs_tabela)} tabelas")

    return todos_texto, todas_tabelas


# ==========================================
# 2. SPLIT — só o texto narrativo é fatiado
# ==========================================
def split_tabela_grande(doc: Document, limite: int = 3000) -> list[Document]:
    """Tabelas > limite de caracteres são divididas repetindo o cabeçalho em cada parte."""
    linhas = doc.page_content.split("\n")
    if len(doc.page_content) <= limite or len(linhas) < 4:
        return [doc]

    cabecalho = linhas[0] + "\n" + linhas[1]  # linha de colunas + linha separadora "---"
    corpo = linhas[2:]

    partes, bloco_atual = [], []
    tamanho_atual = len(cabecalho)

    for linha in corpo:
        if tamanho_atual + len(linha) > limite and bloco_atual:
            conteudo = cabecalho + "\n" + "\n".join(bloco_atual)
            partes.append(Document(page_content=conteudo, metadata=dict(doc.metadata)))
            bloco_atual, tamanho_atual = [], len(cabecalho)
        bloco_atual.append(linha)
        tamanho_atual += len(linha)

    if bloco_atual:
        conteudo = cabecalho + "\n" + "\n".join(bloco_atual)
        partes.append(Document(page_content=conteudo, metadata=dict(doc.metadata)))

    for i, parte in enumerate(partes):
        parte.metadata["tabela_parte"] = f"{i + 1}/{len(partes)}"

    return partes
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


# 12/08/2026 - 14:55:06


## EXECUCAO
### Carregar documento

In [8]:
OUTPUT_DOCUMENTS_DIR

PosixPath('data/fonte')

In [4]:
# ==========================================
# EXECUÇÃO
# ==========================================
print('#', time.strftime("%d/%m/%Y - %H:%M:%S"))
documentos_texto, documentos_tabela = carregar_documentos(OUTPUT_DOCUMENTS_DIR)
print(f"✔ documento carregado | Texto: {len(documentos_texto)} páginas | Tabelas: {len(documentos_tabela)}")
print('#', time.strftime("%d/%m/%Y - %H:%M:%S"))



# 12/08/2026 - 14:55:09
✔ 0 PDF(s) encontrado(s) em data/fonte
✔ documento carregado | Texto: 0 páginas | Tabelas: 0
# 12/08/2026 - 14:55:09


### Split

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,        # mantido igual ao pipeline original
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "!", "?", " "],
    length_function=len,
)
split_texto = text_splitter.split_documents(documentos_texto)

split_tabelas = []
for doc in documentos_tabela:
    split_tabelas.extend(split_tabela_grande(doc))

split_documents = split_texto + split_tabelas

for i, split in enumerate(split_documents):
    split.metadata.update({
        "chunk_id": i,
        "chunk_total": len(split_documents),
        "posicao": f"{i / len(split_documents) * 100:.0f}%",
    })

print("# Documento fatiado em:", time.strftime("%d/%m/%Y - %H:%M:%S"))
print(" chunk_total (texto)  :", len(split_texto))
print(" chunk_total (tabelas):", len(split_tabelas))
print(" chunk_total (geral)  :", len(split_documents))
print("--------------------")



# Documento fatiado em: 12/08/2026 - 14:55:19
 chunk_total (texto)  : 0
 chunk_total (tabelas): 0
 chunk_total (geral)  : 0
--------------------


### Embedding

In [ ]:
# ==========================================
# 3. EMBEDDING E INDEXAÇÃO (igual ao pipeline original)
# ==========================================
model_embedding = 'intfloat/multilingual-e5-small'
embedding_e5 = HuggingFaceEmbeddings(
    model_name=model_embedding,
    model_kwargs={"device": "cpu", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "}
)

batch_size = 150
total = len(split_documents)

print(f"\n#Iniciando criação do banco: {total} chunks")
print(f"#Início: {time.strftime('%H:%M:%S')}\n")



### vectorstore building

In [ ]:
vectorstore = Chroma.from_documents(
    documents=split_documents[:batch_size],
    embedding=embedding_e5,
    persist_directory=str(VECTORSTORE_DIR),
    collection_metadata={"hnsw:space": "cosine"}
)

for i in tqdm(range(batch_size, total, batch_size), desc="Indexando chunks"):
    lote = split_documents[i:i + batch_size]
    vectorstore.add_documents(lote)
    print(f"  Lote {i // batch_size + 1}/{(total // batch_size) + 1} | "
          f"Chunks {i}-{min(i + batch_size, total)} | "
          f"{time.strftime('%H:%M:%S')}")

print(f"\n#Banco criado com {vectorstore._collection.count()} chunks.")
print(f"#Finalizado em: {time.strftime('%H:%M:%S')}")